In [ ]:
# STAGE 2 / STEP 0 - setup (CPU only: this notebook reads the logits and labels saved by the previous notebook, no GPU and no images needed)
import os, json, hashlib, datetime, shutil, math, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata, spearmanr
from scipy.optimize import minimize_scalar
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from tqdm import tqdm
warnings.filterwarnings("ignore", category=RuntimeWarning)

OUT = r"G:\runs"; RS = rf"{OUT}\fair_ens"; VW = rf"{OUT}\vinbig"; S2 = rf"{OUT}\stage2"; os.makedirs(S2, exist_ok=True)   # RS: inputs from the previous notebook | S2: everything created here
SEEDS, L = [0, 1, 2], 6
LABELS6 = ["Cardiomegaly", "Edema", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]
FIND5 = ["Cardiomegaly", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]; IDX5 = [LABELS6.index(l) for l in FIND5]
ARCHS = ["DenseNet", "ConvNeXt"]
plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"], "mathtext.fontset": "stix", "font.size": 8,
    "axes.labelsize": 8, "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7, "axes.linewidth": 0.6,
    "lines.linewidth": 1.0, "grid.linewidth": 0.4, "figure.dpi": 150, "pdf.fonttype": 42})
COL1, COL2 = 3.5, 7.16
def save_fig(fig, name):
    fig.savefig(rf"{S2}\{name}.pdf", bbox_inches="tight"); fig.savefig(rf"{S2}\{name}.png", dpi=600, bbox_inches="tight"); plt.show()
print("inputs:", RS, "| outputs:", S2)

In [ ]:
# STAGE 2 / STEP 1 - the analysis plan. RUN THIS BEFORE YOU LOOK AT ANY NEW RESULT. It is written to disk with a time stamp and a hash, and loaded unchanged on every later run.
PLAN = dict(
    version=1,
    claim="Averaging the logits of one model per source hospital (cross-source ensemble) gives a higher AUROC on an unseen test site than (i) one model trained on the pooled data, "
          "(ii) two pooled models (same inference cost), and (iii) a same-hospital 2-model ensemble.",
    primary_endpoint="mean AUROC over Cardiomegaly, Consolidation, Atelectasis, Pneumothorax, Pleural Effusion",
    secondary_endpoints=["sensitivity at 90% specificity (same 5 findings)", "AUPRC", "ECE"],
    archs=ARCHS, targets=["A", "B", "C"], comparators=["pooled", "pooled x2", "same-source (first source)", "same-source (second source)"],
    delta=0.01, alpha=0.05, correction="Holm over all primary contrasts", B=2000, seed=0,
    cell_pass="the cross-source ensemble beats ALL four comparators (positive difference, Holm-adjusted p < alpha) AND beats pooled training by at least delta",
    replication="PASS if >= 4 of the 6 (architecture x target) cells pass, with at least one passing cell in each architecture and in at least 2 different targets; PARTIAL if >= 1 cell passes; FAIL otherwise",
    ensembling_only_rule="if the cross-source ensemble beats pooled training AND pooled x2 (Holm-adjusted, positive) in >= 4 cells but does not beat both same-source ensembles in most of those cells, report: ensembling helps, hospital diversity is not shown",
    reverse_rule="if pooled training beats the cross-source ensemble (negative difference, Holm-adjusted p < alpha) in >= 4 cells, report pooled training as the better option",
    mechanism=dict(H1="cross-source member pairs are more diverse (lower logit correlation) than same-source pairs; the bootstrap CI excludes 0 in a majority of (architecture x target) cells",
                   H2="the gain of a 2-member ensemble over its mean member AUROC increases with member diversity (Spearman > 0, CI excludes 0)",
                   H3="ensembling does not repair calibration; Platt scaling with k = 500 target labels lowers ECE"),
    robustness=["per seed", "leave-one-finding-out", "aggregation rule (logit / probability / rank)", "label rule on VinBigData (any radiologist)", "minimal detectable effect"],
)
PF = rf"{S2}\analysis_plan_v1.json"
phash = lambda p: hashlib.sha256(json.dumps(p, sort_keys=True).encode()).hexdigest()[:16]
if os.path.exists(PF):
    saved = json.load(open(PF)); PLAN = saved["plan"]
    assert phash(PLAN) == saved["hash"], "the locked plan file was edited after it was written: restore it, or start a new version on purpose"
    print(f"plan LOADED from disk (locked {saved['created']}, hash {saved['hash']}). Edits to the PLAN in this cell are ignored.")
else:
    json.dump(dict(created=str(datetime.datetime.now()), hash=phash(PLAN), plan=PLAN), open(PF, "w"), indent=2); print(f"plan LOCKED now, hash {phash(PLAN)}")
B, DELTA, ALPHA = PLAN["B"], PLAN["delta"], PLAN["alpha"]
for k in ["claim", "primary_endpoint", "cell_pass", "replication", "ensembling_only_rule", "reverse_rule"]: print(f"\n{k}:\n  {PLAN[k]}")
print("\nthis only documents the plan: nothing here can prove you did not look at results first, so say honestly in the paper when the plan was fixed.")

In [ ]:
# STAGE 2 / STEP 2 - load labels and the logits of every model on the three test sites, build the model families (same definitions as the previous notebook)
TG = {"A": "NIH", "B": "CheXpert", "C": "VinBigData"}
SRCS = {"A": ("B", "C"), "B": ("A", "C"), "C": ("A", "B")}; POOL = {"A": "B+C", "B": "A+C", "C": "A+B"}; TS_ALL = ["A", "B", "C", "A+B", "B+C", "A+C"]
nih = pd.read_csv(r"G:\nih_split.csv"); off = nih.official.values == "test"
cx = pd.read_csv(r"G:\chexpert_final_clean.csv"); cxt = cx[cx.split == "test"].reset_index(drop=True)
vb = pd.read_csv(rf"{VW}\vinbig_image_labels.csv"); NVB = len(vb)
Yd = {"A": nih[LABELS6].values.astype(int)[off], "B": cxt[LABELS6].values.astype(int), "C": np.zeros((NVB, L), int)}
Yany = np.zeros((NVB, L), int)
for f in FIND5: Yd["C"][:, LABELS6.index(f)] = vb[f].values; Yany[:, LABELS6.index(f)] = vb[f + "_any"].values
grp = {"A": pd.factorize(nih.patient_id.values[off])[0], "B": pd.factorize(cxt.patient_id.values)[0], "C": None}      # patient index per image (None: every VinBigData image is a different person)
VALID_C = np.array([l != "Edema" for l in LABELS6])
lpath = lambda a, ts, s: rf"{RS}\logits_{a}_{ts.replace('+', 'p')}_erm_nat_seed{s}.npz"
LGT = {T: {} for T in TG}; AVAIL = set()
for a in ARCHS:
    for ts in TS_ALL:
        if all(os.path.exists(lpath(a, ts, s)) for s in SEEDS):
            AVAIL.add((a, ts))
            for s in SEEDS:
                z = np.load(lpath(a, ts, s))
                for T in TG: LGT[T].setdefault((a, ts), {})[s] = z[T]
for T in TG: assert all(len(Yd[T]) == v[0].shape[0] for v in LGT[T].values()), f"label / logit length mismatch on {T}"
print("model sets ready (all 3 seeds):", sorted(AVAIL)); print("missing:", sorted({(a, ts) for a in ARCHS for ts in TS_ALL} - AVAIL))

def member(T, X, ts, s):
    z = LGT[T][(X, ts)][s % len(SEEDS)]; v = VALID_C if ts == "C" else np.ones(L, bool); return z, np.broadcast_to(v, z.shape)
def ens(ms):                                                       # average of logits; a VinBigData-only model has no Edema output and is left out of that finding
    Z = np.stack([z for z, _ in ms]); M = np.stack([v for _, v in ms]).astype(float); return (Z * M).sum(0) / np.maximum(M.sum(0), 1)
def fam_defs(T):
    s1, s2 = SRCS[T]; P = POOL[T]; D = {}
    for X in ARCHS:
        D[f"{X}: {s1}-only"] = [(X, s1, 0)]; D[f"{X}: {s2}-only"] = [(X, s2, 0)]
        D[f"{X}: pooled {P}"] = [(X, P, 0)]; D[f"{X}: pooled {P} x2 seeds"] = [(X, P, 0), (X, P, 1)]
        D[f"{X}: ensemble {s1} + {s2}"] = [(X, s1, 0), (X, s2, 0)]
        D[f"{X}: ensemble {s1} + {s1}"] = [(X, s1, 0), (X, s1, 1)]; D[f"{X}: ensemble {s2} + {s2}"] = [(X, s2, 0), (X, s2, 1)]
    for s_ in (s1, s2): D[f"{s_}-only: DenseNet + ConvNeXt"] = [("DenseNet", s_, 0), ("ConvNeXt", s_, 0)]
    D[f"4 models: {s1}-only + {s2}-only, both architectures"] = [(X, s_, 0) for X in ARCHS for s_ in (s1, s2)]
    D[f"pooled {P}: DenseNet + ConvNeXt"] = [(X, P, 0) for X in ARCHS]; D[f"pooled {P} x4 (2 archs x 2 seeds)"] = [(X, P, o) for X in ARCHS for o in (0, 1)]
    return {n: mm for n, mm in D.items() if all((x, ts_) in AVAIL for x, ts_, _ in mm)}
FD = {T: fam_defs(T) for T in TG}
FL = {T: {n: {s: ens([member(T, X, ts, s + o) for X, ts, o in mm]) for s in SEEDS} for n, mm in FD[T].items()} for T in TG}
print({TG[T]: len(FL[T]) for T in TG}, "model families per test site")

In [ ]:
# STAGE 2 / STEP 3 - fast cluster-bootstrap engine. Resampling patients (NIH, CheXpert) or images (VinBigData) is done with multiplicity weights, so each AUROC costs one sort-once pass.
class Engine:
    """AUROC and sensitivity at 90% specificity of any family under bootstrap weights c (one weight per image). Tied scores (common with half-precision logits) get half credit, exactly like the rank-based AUROC."""
    def __init__(self, Y): self.Y, self.cache = Y, {}
    def _prep(self, T, name, s, j):
        k = (T, name, s, j)
        if k not in self.cache:
            sc = FL[T][name][s][:, j]; o = np.argsort(sc, kind="stable"); ss = sc[o]
            gid = np.cumsum(np.r_[True, ss[1:] != ss[:-1]]) - 1                               # images with the same score share one group id
            self.cache[k] = (o, self.Y[o, j].astype(float), gid, int(gid[-1]) + 1)
        return self.cache[k]
    def metrics(self, T, name, c):                                 # -> (AUROC[5], sens@90spec[5]), averaged over the 3 seeds, one value per finding
        au, se = np.zeros(5), np.zeros(5)
        for jj, j in enumerate(IDX5):
            a = s_ = 0.0
            for s in SEEDS:
                o, ys, gid, ng = self._prep(T, name, s, j); cs = c[o]; pos = cs * ys; neg = cs - pos; P, Nn = pos.sum(), neg.sum()
                gneg = np.bincount(gid, weights=neg, minlength=ng); gpos = np.bincount(gid, weights=pos, minlength=ng); before = np.cumsum(gneg) - gneg
                a += (gpos * (before + 0.5 * gneg)).sum() / (P * Nn)
                g = np.searchsorted(np.cumsum(gneg), 0.9 * Nn); s_ += gpos[g + 1:].sum() / P         # best sensitivity with specificity >= 90%
            au[jj], se[jj] = a / len(SEEDS), s_ / len(SEEDS)
        return au, se
def fauc(y, s):
    y = np.asarray(y).astype(bool); r = rankdata(s); n1 = y.sum(); n0 = len(y) - n1
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
def counts(T, rng):
    g = grp[T]; N = len(Yd[T])
    if g is None: return np.bincount(rng.integers(0, N, N), minlength=N).astype(float)
    G = g.max() + 1; return np.bincount(rng.integers(0, G, G), minlength=G)[g].astype(float)

ENG = {T: Engine(Yd[T]) for T in TG}
from sklearn.metrics import roc_curve
for T in TG:                                                       # self-tests against independent references (rank-based AUROC, sklearn ROC curve, and row duplication for integer weights)
    n0 = next(iter(FL[T])); Y_ = Yd[T]; ones = np.ones(len(Y_)); fast_auc, fast_sens = ENG[T].metrics(T, n0, ones)
    slow_auc = np.array([np.mean([fauc(Y_[:, j], FL[T][n0][s][:, j]) for s in SEEDS]) for j in IDX5])
    def sens_ref(y, sc): fpr, tpr, _ = roc_curve(y, sc, drop_intermediate=False); return tpr[fpr <= 0.1].max()
    slow_sens = np.array([np.mean([sens_ref(Y_[:, j], FL[T][n0][s][:, j]) for s in SEEDS]) for j in IDX5])
    assert np.abs(fast_auc - slow_auc).max() < 1e-9, f"AUROC check failed on {T}: {np.abs(fast_auc - slow_auc).max()}"
    assert np.abs(fast_sens - slow_sens).max() < 1e-9, f"sensitivity check failed on {T}: {np.abs(fast_sens - slow_sens).max()}"
    w = np.random.default_rng(0).integers(0, 3, len(Y_)).astype(float); rep = np.repeat(np.arange(len(Y_)), w.astype(int)); j0 = IDX5[0]      # weights 0/1/2 equal duplicating or dropping rows
    fw = ENG[T].metrics(T, n0, w)[0][0]; sw = np.mean([fauc(Y_[rep, j0], FL[T][n0][s][rep, j0]) for s in SEEDS]); assert abs(fw - sw) < 1e-9, f"weighted AUROC check failed on {T}: {abs(fw - sw)}"
    n_tied = len(Y_) - len(np.unique(FL[T][n0][0][:, j0])); print(f"engine check passed on {TG[T]} (images sharing a score with another image: {n_tied})")
BOOT, POINT = {}, {}
for T in TG:
    rng = np.random.default_rng(PLAN["seed"]); names = list(FL[T]); arr = {n: (np.zeros((B, 5)), np.zeros((B, 5))) for n in names}      # the same resamples are used for every family: all comparisons are paired
    for b in tqdm(range(B), desc=f"bootstrap {TG[T]} (B = {B})"):
        c = counts(T, rng)
        for n in names: a, s_ = ENG[T].metrics(T, n, c); arr[n][0][b], arr[n][1][b] = a, s_
    BOOT[T] = arr; POINT[T] = {n: ENG[T].metrics(T, n, np.ones(len(Yd[T]))) for n in names}
print("bootstrap done | resamples with an undefined finding (no positives drawn):", int(sum(np.isnan(BOOT[T][next(iter(BOOT[T]))][0]).any(1).sum() for T in TG)))

In [ ]:
# STAGE 2 / STEP 4 - the primary contrasts, Holm correction and the decision gates written in the plan
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for r, i in enumerate(o): run = max(run, (m - r) * p[i]); adj[i] = min(run, 1.0)
    return adj
rows = []
for T in TG:
    for X in ARCHS:
        s1, s2 = SRCS[T]; P = POOL[T]; cross = f"{X}: ensemble {s1} + {s2}"
        comps = {"pooled": f"{X}: pooled {P}", "pooled x2": f"{X}: pooled {P} x2 seeds", f"same-source ({s1})": f"{X}: ensemble {s1} + {s1}", f"same-source ({s2})": f"{X}: ensemble {s2} + {s2}"}
        if cross not in FL[T] or any(v not in FL[T] for v in comps.values()): print(f"not evaluable (models missing): {X}, test site {TG[T]}"); continue
        for cn, nm in comps.items():
            d = np.nanmean(BOOT[T][cross][0], 1) - np.nanmean(BOOT[T][nm][0], 1); d = d[np.isfinite(d)]; lo, hi = np.percentile(d, [2.5, 97.5])      # a resample with no positive of a rare finding skips that finding
            p = (min((d <= 0).sum(), (d >= 0).sum()) * 2 + 1) / (len(d) + 1)
            rows.append(dict(target=T, test_site=TG[T], arch=X, comparator=cn, diff=float(d.mean()), lo=lo, hi=hi, se=float(d.std(ddof=1)), p_raw=min(p, 1.0)))
PC = pd.DataFrame(rows); PC["p_holm"] = holm(PC.p_raw.values)
PC["beats"] = (PC["diff"] > 0) & (PC.p_holm < ALPHA); PC["loses"] = (PC["diff"] < 0) & (PC.p_holm < ALPHA)
cells = []
for (T, X), g in PC.groupby(["target", "arch"]):
    ok = g.beats.all() and len(g) == 4 and float(g[g.comparator == "pooled"]["diff"].iloc[0]) >= DELTA
    lose = bool(g[g.comparator == "pooled"].loses.iloc[0])
    gd = g.set_index("comparator"); bp, bp2 = bool(gd.loc["pooled", "beats"]), bool(gd.loc["pooled x2", "beats"]); bsame = bool(g[g.comparator.str.startswith("same")].beats.all())
    cells.append(dict(target=T, test_site=TG[T], arch=X, passes=bool(ok), pooled_wins=lose, diff_vs_pooled=float(g[g.comparator == "pooled"]["diff"].iloc[0]), n_comparators_beaten=int(g.beats.sum()),
                      beats_pooled_and_x2=bp and bp2, beats_both_same_source=bsame))
CELLS = pd.DataFrame(cells); n_pass, n_lose = int(CELLS.passes.sum()), int(CELLS.pooled_wins.sum())
archs_ok = CELLS[CELLS.passes].arch.nunique() if n_pass else 0; targets_ok = CELLS[CELLS.passes].target.nunique() if n_pass else 0
VERDICT = "PASS" if (n_pass >= 4 and archs_ok == len(ARCHS) and targets_ok >= 2) else ("PARTIAL" if n_pass >= 1 else "FAIL")
REVERSE = n_lose >= 4
ens_cells = CELLS[CELLS.beats_pooled_and_x2]; ENS_ONLY = bool(len(ens_cells) >= 4 and (~ens_cells.beats_both_same_source).mean() > 0.5 and VERDICT != "PASS")
TIER = "REVERSED" if REVERSE else ("PASS" if VERDICT == "PASS" else ("ENSEMBLING_ONLY" if ENS_ONLY else VERDICT))
display(PC.drop(columns=["target"]).round(4)); display(CELLS.round(4)); PC.to_csv(rf"{S2}\primary_contrasts.csv", index=False); CELLS.to_csv(rf"{S2}\primary_cells.csv", index=False)
print(f"\ncells passing: {n_pass} of {len(CELLS)} | architectures with a pass: {archs_ok} | test sites with a pass: {targets_ok}")
print(f"VERDICT FOR THE HEADLINE CLAIM: {TIER}   (plan ladder: PASS / ENSEMBLING_ONLY / PARTIAL / FAIL / REVERSED)")
if TIER == "ENSEMBLING_ONLY": print("the ensemble beats pooled training but a same-hospital ensemble does as well: the gain comes from ensembling, not from hospital diversity")
if REVERSE: print("pooled training is significantly better in >= 4 cells: report it as the better option")
print("minimal detectable difference (about 2.8 x standard error, 80% power) per contrast: median", round(2.8 * PC.se.median(), 4), "| the plan requires", DELTA)

In [ ]:
# STAGE 2 / STEP 5 - mechanism. H1: do models trained on different hospitals disagree more than models trained on the same hospital? H2: does an ensemble gain more when its members disagree more?
def wcorr(a, b, c):
    w = c / c.sum(); ma, mb = (w * a).sum(), (w * b).sum(); va, vb_ = (w * (a - ma) ** 2).sum(), (w * (b - mb) ** 2).sum()
    return (w * (a - ma) * (b - mb)).sum() / np.sqrt(va * vb_)
KINDS = {"cross": "different hospitals", "same1": "same hospital (first source)", "same2": "same hospital (second source)"}
def pair(T, X, kind, s):
    s1, s2 = SRCS[T]; F_ = FL[T]
    if kind == "cross": return F_[f"{X}: {s1}-only"][s], F_[f"{X}: {s2}-only"][s], F_[f"{X}: ensemble {s1} + {s2}"][s]
    if kind == "same1": return F_[f"{X}: {s1}-only"][s], F_[f"{X}: {s1}-only"][(s + 1) % 3], F_[f"{X}: ensemble {s1} + {s1}"][s]
    return F_[f"{X}: {s2}-only"][s], F_[f"{X}: {s2}-only"][(s + 1) % 3], F_[f"{X}: ensemble {s2} + {s2}"][s]
B1 = 500; rows, pts = [], []
for T in TG:
    for X in ARCHS:
        s1, s2 = SRCS[T]
        if f"{X}: ensemble {s1} + {s2}" not in FL[T] or f"{X}: ensemble {s1} + {s1}" not in FL[T] or f"{X}: ensemble {s2} + {s2}" not in FL[T]: continue
        rng = np.random.default_rng(PLAN["seed"] + 1); dd = np.zeros(B1); dc = np.zeros(B1)
        for b in range(B1):
            c = counts(T, rng); D = {k: np.mean([1 - wcorr(pair(T, X, k, s)[0][:, j], pair(T, X, k, s)[1][:, j], c) for s in SEEDS for j in IDX5]) for k in KINDS}
            dc[b], dd[b] = D["cross"], D["cross"] - (D["same1"] + D["same2"]) / 2
        lo, hi = np.percentile(dd, [2.5, 97.5])
        rows.append(dict(test_site=TG[T], arch=X, diversity_cross=dc.mean(), diversity_diff_cross_minus_same=dd.mean(), lo=lo, hi=hi, excludes_0=bool(lo > 0 or hi < 0), cross_more_diverse=bool(lo > 0)))
        for k in KINDS:                                                # point estimates per finding for the gain-versus-diversity relation
            for j, f in zip(IDX5, FIND5):
                dv, gn = [], []
                for s in SEEDS:
                    m1, m2, e = pair(T, X, k, s); dv.append(1 - np.corrcoef(m1[:, j], m2[:, j])[0, 1]); gn.append(fauc(Yd[T][:, j], e[:, j]) - (fauc(Yd[T][:, j], m1[:, j]) + fauc(Yd[T][:, j], m2[:, j])) / 2)
                pts.append(dict(test_site=TG[T], arch=X, pair=k, finding=f, diversity=np.mean(dv), gain=np.mean(gn)))
H1, PTS = pd.DataFrame(rows), pd.DataFrame(pts)
display(H1.round(4)); H1.to_csv(rf"{S2}\mechanism_H1_diversity.csv", index=False); PTS.to_csv(rf"{S2}\mechanism_H2_points.csv", index=False)
rho = spearmanr(PTS.diversity, PTS.gain)[0]; rng = np.random.default_rng(3); bs = [spearmanr(*PTS.sample(len(PTS), replace=True, random_state=int(rng.integers(1e9)))[["diversity", "gain"]].values.T)[0] for _ in range(1000)]
RHO = (rho, *np.nanpercentile(bs, [2.5, 97.5]))
H1_PASS = bool(len(H1) and H1.cross_more_diverse.mean() > 0.5); H2_PASS = bool(RHO[1] > 0)
print(f"H1 (cross-hospital pairs are more diverse in a majority of cells): {'SUPPORTED' if H1_PASS else 'NOT supported'} ({int(H1.cross_more_diverse.sum())} of {len(H1)} cells)")
print(f"H2 (gain grows with diversity): Spearman rho = {RHO[0]:.3f}, 95% CI [{RHO[1]:.3f}, {RHO[2]:.3f}] over {len(PTS)} (test site x architecture x pair type x finding) points -> {'SUPPORTED' if H2_PASS else 'NOT supported'}")
print("note: the points share models and test images, so the interval is optimistic; read H2 as supporting evidence only.")

In [ ]:
# STAGE 2 / STEP 6 - robustness of the headline contrast (cross-source ensemble minus pooled training)
mean5 = lambda T, name, s: np.mean([fauc(Yd[T][:, j], FL[T][name][s][:, j]) for j in IDX5])
rows = []
for T in TG:
    for X in ARCHS:
        s1, s2 = SRCS[T]; P = POOL[T]; cr, po, p2 = f"{X}: ensemble {s1} + {s2}", f"{X}: pooled {P}", f"{X}: pooled {P} x2 seeds"
        if cr not in FL[T] or po not in FL[T]: continue
        per_seed = [mean5(T, cr, s) - mean5(T, po, s) for s in SEEDS]                                     # R1: does every seed agree?
        pf = POINT[T][cr][0] - POINT[T][po][0]; lofo = [np.delete(pf, i).mean() for i in range(5)]       # R2: does one finding drive the result?
        rows.append(dict(test_site=TG[T], arch=X, diff_all=pf.mean(), seeds_positive=f"{sum(d > 0 for d in per_seed)}/3", diff_by_seed=" / ".join(f"{d:+.3f}" for d in per_seed),
                         lofo_min=min(lofo), dropped_when_min=FIND5[int(np.argmin(lofo))], findings_positive=f"{int((pf > 0).sum())}/5"))
R12 = pd.DataFrame(rows); display(R12.round(4)); R12.to_csv(rf"{S2}\robustness_seed_finding.csv", index=False)

def scores(zs, how, j):                                                                                  # R3: the aggregation rule must not decide the conclusion
    if how == "logit": return np.mean([z[:, j] for z in zs], 0)
    if how == "probability": return np.mean([1 / (1 + np.exp(-z[:, j])) for z in zs], 0)
    return np.mean([rankdata(z[:, j]) / len(z) for z in zs], 0)
rows = []
for T in TG:
    for X in ARCHS:
        s1, s2 = SRCS[T]; P = POOL[T]
        if f"{X}: ensemble {s1} + {s2}" not in FL[T] or f"{X}: pooled {P} x2 seeds" not in FL[T]: continue
        for how in ["logit", "probability", "rank"]:
            a = np.mean([np.mean([fauc(Yd[T][:, j], scores([FL[T][f"{X}: {s1}-only"][s], FL[T][f"{X}: {s2}-only"][s]], how, j)) for j in IDX5]) for s in SEEDS])
            b2 = np.mean([np.mean([fauc(Yd[T][:, j], scores([FL[T][f"{X}: pooled {P}"][s], FL[T][f"{X}: pooled {P}"][(s + 1) % 3]], how, j)) for j in IDX5]) for s in SEEDS])
            rows.append(dict(test_site=TG[T], arch=X, aggregation=how, cross_source=a, pooled_x2=b2, cross_minus_pooled=a - np.mean([mean5(T, f"{X}: pooled {P}", s) for s in SEEDS]), cross_minus_pooled_x2=a - b2))
R3 = pd.DataFrame(rows); display(R3.round(4)); R3.to_csv(rf"{S2}\robustness_aggregation.csv", index=False)

# R4: label rule on VinBigData (ground truth = any radiologist instead of at least 2 of 3)
ENG_ANY = Engine(Yany); rng = np.random.default_rng(7); B4 = min(B, 1000); rows = []
need = {X: [f"{X}: ensemble A + B", f"{X}: pooled A+B", f"{X}: pooled A+B x2 seeds"] for X in ARCHS}
use = [X for X in ARCHS if all(n in FL["C"] for n in need[X])]
acc = {(X, n): np.zeros(B4) for X in use for n in need[X]}
for b in tqdm(range(B4), desc="R4 label rule"):
    c = counts("C", rng)
    for X in use:
        for n in need[X]: acc[(X, n)][b] = np.nanmean(ENG_ANY.metrics("C", n, c)[0])
rows = []
for X in use:
    for lab_, other in [("pooled", need[X][1]), ("pooled x2", need[X][2])]:
        d = acc[(X, need[X][0])] - acc[(X, other)]; d = d[np.isfinite(d)]; lo, hi = np.percentile(d, [2.5, 97.5]); rows.append(dict(arch=X, comparator=lab_, diff_any_radiologist=d.mean(), lo=lo, hi=hi, excludes_0=bool(lo > 0 or hi < 0)))
R4 = pd.DataFrame(rows); display(R4.round(4)); R4.to_csv(rf"{S2}\robustness_label_rule.csv", index=False)
print("R5: minimal detectable difference per primary contrast (2.8 x SE) -> median", round(2.8 * PC.se.median(), 4), ", largest", round(2.8 * PC.se.max(), 4), "| smaller true effects cannot be separated from zero with these test sets")

In [ ]:
# STAGE 2 / STEP 7 - secondary endpoints (sensitivity at 90% specificity, AUPRC, ECE) and H3 (calibration after ensembling, and Platt scaling with k labelled target images)
sig = lambda z: 1 / (1 + np.exp(-z))
def ece_of(y, p, nb=10):
    bi = np.clip((p * nb).astype(int), 0, nb - 1); e = 0.0
    for k in range(nb):
        mk = bi == k
        if mk.any(): e += mk.mean() * abs(p[mk].mean() - y[mk].mean())
    return e
rows = []
for T in TG:
    for X in ARCHS:
        s1, s2 = SRCS[T]; P = POOL[T]; cr, po, p2 = f"{X}: ensemble {s1} + {s2}", f"{X}: pooled {P}", f"{X}: pooled {P} x2 seeds"
        if not all(n in FL[T] for n in (cr, po, p2)): continue
        for nm, lab_ in [(cr, "cross-source ensemble"), (po, "pooled"), (p2, "pooled x2")]:
            sc = [sig(FL[T][nm][s]) for s in SEEDS]
            rows.append(dict(test_site=TG[T], arch=X, model=lab_, AUROC=POINT[T][nm][0].mean(), sens_at_90spec=POINT[T][nm][1].mean(),
                             AUPRC=np.mean([np.mean([average_precision_score(Yd[T][:, j], p[:, j]) for j in IDX5]) for p in sc]), ECE=np.mean([np.mean([ece_of(Yd[T][:, j], p[:, j]) for j in IDX5]) for p in sc])))
SEC = pd.DataFrame(rows); display(SEC.round(4)); SEC.to_csv(rf"{S2}\secondary_endpoints.csv", index=False)
rows = []
for T in TG:
    for X in ARCHS:
        s1, s2 = SRCS[T]; P = POOL[T]; cr = f"{X}: ensemble {s1} + {s2}"
        for lab_, other in [("pooled", f"{X}: pooled {P}"), ("pooled x2", f"{X}: pooled {P} x2 seeds")]:
            if cr not in FL[T] or other not in FL[T]: continue
            d = np.nanmean(BOOT[T][cr][1], 1) - np.nanmean(BOOT[T][other][1], 1); d = d[np.isfinite(d)]; lo, hi = np.percentile(d, [2.5, 97.5])
            rows.append(dict(test_site=TG[T], arch=X, comparator=lab_, diff_sens_at_90spec=d.mean(), lo=lo, hi=hi, excludes_0=bool(lo > 0 or hi < 0)))
SENS = pd.DataFrame(rows); display(SENS.round(4)); SENS.to_csv(rf"{S2}\secondary_sensitivity_contrasts.csv", index=False)

def fit_platt(z, y):
    lr = LogisticRegression(C=1e6, max_iter=500).fit(z.reshape(-1, 1), y); return float(np.clip(lr.coef_[0, 0], 0.1, 5.0)), float(lr.intercept_[0])
GRID = np.linspace(0.02, 0.95, 94)
def best_thr(y, p):
    pr = p[:, None] >= GRID[None, :]; y1 = (y == 1)[:, None]; tp = (pr & y1).sum(0); fp = pr.sum(0) - tp; fn = y1.sum() - tp
    return GRID[np.argmax(2 * tp / np.maximum(2 * tp + fp + fn, 1))]
def f1_at(y, p, t):
    pr = p >= t; y1 = y == 1; tp = (pr & y1).sum(); dn = 2 * tp + (pr & ~y1).sum() + (~pr & y1).sum(); return 2 * tp / dn if dn else 0.0
def halves(T, rng):                                                # pool half / evaluation half, split by patient (NIH, CheXpert) or by image (VinBigData)
    g = grp[T]; N = len(Yd[T])
    if g is None: pm = rng.permutation(N); return pm[:N // 2], pm[N // 2:]
    G = g.max() + 1; inp = np.zeros(G, bool); inp[rng.permutation(G)[:G // 2]] = True; m = inp[g]; return np.where(m)[0], np.where(~m)[0]
KS, DRAWS = [0, 100, 250, 500, 1000, 2500], 10; rows = []
for T in TG:
    for X in ARCHS:
        s1, s2 = SRCS[T]; P = POOL[T]; mods = {"pooled": f"{X}: pooled {P}", "cross-source ensemble": f"{X}: ensemble {s1} + {s2}"}
        if not all(n in FL[T] for n in mods.values()): continue
        for lab_, nm in mods.items():
            for s in SEEDS:
                Z = FL[T][nm][s]
                for dr in range(DRAWS):
                    pool, ev = halves(T, np.random.default_rng([s, dr]))
                    for K in KS:
                        if K == 0 and dr > 0: continue
                        ip = pool[:K]; es, fs = [], []
                        for j in IDX5:
                            yp, ze, ye = Yd[T][ip, j], Z[ev, j], Yd[T][ev, j]; a, b_, ok = 1.0, 0.0, False
                            if K and yp.sum() >= 5 and (len(yp) - yp.sum()) >= 5: a, b_ = fit_platt(Z[ip, j], yp); ok = True
                            pe = sig(a * ze + b_); es.append(ece_of(ye, pe))
                            if K and ok: fs.append(f1_at(ye, pe, best_thr(yp, sig(a * Z[ip, j] + b_))))
                        rows.append(dict(test_site=TG[T], arch=X, model=lab_, seed=s, K=K, draw=dr, ece=np.mean(es), f1=np.mean(fs) if fs else np.nan, n_fit=len(fs)))
RC = pd.DataFrame(rows); RC.to_csv(rf"{S2}\recalibration_raw.csv", index=False)
RCS = RC.groupby(["test_site", "arch", "model", "K"])[["ece", "f1", "n_fit"]].mean().reset_index()
t = RCS[RCS.K.isin([0, 100, 500, 2500])].pivot_table(index=["test_site", "arch", "model"], columns="K", values="ece"); display(t.round(4)); t.to_csv(rf"{S2}\recalibration_ece.csv")
t2 = RCS[RCS.K.isin([100, 500, 2500])].pivot_table(index=["test_site", "arch", "model"], columns="K", values="f1"); display(t2.round(4))
e0 = RCS[RCS.K == 0].pivot_table(index=["test_site", "arch"], columns="model", values="ece"); e5 = RCS[RCS.K == 500].pivot_table(index=["test_site", "arch"], columns="model", values="ece")
H3a = bool((e0["cross-source ensemble"] >= e0["pooled"] - 0.005).mean() > 0.5); H3b = bool((e5["cross-source ensemble"] < e0["cross-source ensemble"]).all())
print(f"H3a (the ensemble is not better calibrated than pooled training in a majority of cells): {'SUPPORTED' if H3a else 'NOT supported'}")
print(f"H3b (Platt scaling with 500 labelled images lowers the ensemble's ECE in every cell): {'SUPPORTED' if H3b else 'NOT supported'}")
print("F1 needs a threshold tuned on the labelled pool, so it is only reported for k > 0; findings without 5 positives and 5 negatives in the pool stay uncorrected.")

In [ ]:
# STAGE 2 / STEP 8 - figures
# 8A: the primary contrasts, one panel per (test site x architecture)
fig, axs = plt.subplots(len(TG), len(ARCHS), figsize=(COL2, 6.0), sharex=True)
for r_, T in enumerate(TG):
    for c_, X in enumerate(ARCHS):
        a_ = axs[r_, c_]; g = PC[(PC.target == T) & (PC.arch == X)].reset_index(drop=True)
        if not len(g): a_.axis("off"); continue
        for i, r in g.iterrows():
            col = "C3" if r.beats else ("C0" if r.loses else "0.5"); a_.plot([r.lo, r.hi], [i, i], color="k", lw=1); a_.plot(r["diff"], i, "o", color=col, ms=4)
        cp = CELLS[(CELLS.target == T) & (CELLS.arch == X)].passes.iloc[0]
        a_.axvline(0, color="k", lw=0.6); a_.axvline(DELTA, color="C1", lw=0.8, ls=":"); a_.set_yticks(range(len(g))); a_.set_yticklabels(g.comparator, fontsize=6); a_.invert_yaxis()
        a_.set_title(f"{X}, {TG[T]}: {'PASS' if cp else 'does not pass'}", fontsize=7); a_.grid(True, axis="x", alpha=0.3)
for a_ in axs[-1]: a_.set_xlabel("AUROC of the cross-source ensemble minus comparator")
fig.text(0.5, -0.01, "red = beats the comparator (Holm-adjusted p < 0.05) | blue = loses | grey = not significant | orange dotted line = required effect (delta)", ha="center", fontsize=6)
fig.tight_layout(); save_fig(fig, "fig_s2_primary_contrasts")

# 8B: gate map
ck = [(T, X) for T in TG for X in ARCHS if len(PC[(PC.target == T) & (PC.arch == X)])]
fig, ax = plt.subplots(figsize=(COL2, 0.55 * len(ck) + 1.4)); M = np.full((len(ck), 4), np.nan); S_ = [[""] * 4 for _ in ck]
for i, (T, X) in enumerate(ck):
    g = PC[(PC.target == T) & (PC.arch == X)].reset_index(drop=True)
    for j, r in g.iterrows(): M[i, j] = r["diff"]; S_[i][j] = f"{r['diff']:+.3f}" + ("*" if r.beats else ("†" if r.loses else ""))
v = np.nanmax(np.abs(M)) if np.isfinite(M).any() else 0.05; ax.imshow(M, cmap="coolwarm", vmin=-v, vmax=v, aspect="auto")
for i in range(len(ck)):
    for j in range(4): ax.text(j, i, S_[i][j], ha="center", va="center", fontsize=6.5, color="w" if np.isfinite(M[i, j]) and abs(M[i, j]) > 0.75 * v else "k")
ax.set_xticks(range(4)); ax.set_xticklabels(["pooled", "pooled x2", "same-source (1st)", "same-source (2nd)"], rotation=20, ha="right")
ax.set_yticks(range(len(ck))); ax.set_yticklabels([f"{X}, {TG[T]}  [{'PASS' if CELLS[(CELLS.target == T) & (CELLS.arch == X)].passes.iloc[0] else 'no'}]" for T, X in ck]); ax.grid(False)
ax.set_title(f"Gate map (verdict: {TIER}); * cross-source ensemble significantly better, † significantly worse (Holm)"); fig.tight_layout(); save_fig(fig, "fig_s2_gate_map")

# 8C: mechanism
fig, axs = plt.subplots(1, 2, figsize=(COL2, 3.2))
lab_ = [f"{r.arch[:5]}, {r.test_site}" for r in H1.itertuples()]
axs[0].barh(range(len(H1)), H1.diversity_diff_cross_minus_same, xerr=[H1.diversity_diff_cross_minus_same - H1.lo, H1.hi - H1.diversity_diff_cross_minus_same], color="0.6", edgecolor="k", lw=0.4, capsize=2)
axs[0].axvline(0, color="k", lw=0.6); axs[0].set_yticks(range(len(H1))); axs[0].set_yticklabels(lab_, fontsize=6); axs[0].invert_yaxis(); axs[0].set_xlabel("Diversity, cross-hospital minus same-hospital pairs"); axs[0].set_title("H1: diversity of member pairs")
mk = {"DenseNet": "o", "ConvNeXt": "s"}; pc_ = {"cross": "C3", "same1": "0.5", "same2": "0.75"}
for (a, k), g in PTS.groupby(["arch", "pair"]): axs[1].scatter(g.diversity, g.gain, s=10, marker=mk[a], color=pc_[k], alpha=0.8, label=f"{a[:5]}, {KINDS[k]}")
axs[1].axhline(0, color="k", lw=0.6); axs[1].set_xlabel("Member diversity (1 - logit correlation)"); axs[1].set_ylabel("Ensemble gain in AUROC"); axs[1].legend(frameon=False, fontsize=5, ncol=1, loc="lower right")
axs[1].set_title(f"H2: gain vs diversity\nSpearman {RHO[0]:.2f}, CI [{RHO[1]:.2f}, {RHO[2]:.2f}]", fontsize=7); [a_.grid(True, alpha=0.3) for a_ in axs]
fig.tight_layout(w_pad=3.0); save_fig(fig, "fig_s2_mechanism")

# 8D: robustness
fig, axs = plt.subplots(1, 3, figsize=(COL2, 3.2)); cl = [f"{r.arch[:5]}, {r.test_site}" for r in R12.itertuples()]
for i, r in enumerate(R12.itertuples()):
    axs[0].scatter([float(x) for x in r.diff_by_seed.split(" / ")], [i] * 3, color="C3", s=14); axs[0].plot([r.diff_all], [i], "k|", ms=10)
axs[0].axvline(0, color="k", lw=0.6); axs[0].set_yticks(range(len(cl))); axs[0].set_yticklabels(cl, fontsize=6); axs[0].invert_yaxis(); axs[0].set_xlabel("Difference: seeds red, mean black", fontsize=7); axs[0].set_title("R1: seeds")
axs[1].scatter(R12.diff_all, R12.lofo_min, color="0.3", s=18); lim = [min(R12.diff_all.min(), R12.lofo_min.min()) - 0.01, max(R12.diff_all.max(), R12.lofo_min.max()) + 0.01]; axs[1].plot(lim, lim, "k:", lw=0.6); axs[1].axhline(0, color="k", lw=0.6)
axs[1].set(xlabel="Difference, all 5 findings", ylabel="Smallest difference, one finding dropped", title="R2: findings")
for how, col in [("logit", "C3"), ("probability", "C0"), ("rank", "0.5")]:
    g = R3[R3.aggregation == how]; axs[2].scatter(g.cross_minus_pooled, range(len(g)), color=col, s=14, label=how)
axs[2].axvline(0, color="k", lw=0.6); axs[2].set_yticks(range(len(R3[R3.aggregation == "logit"]))); axs[2].set_yticklabels([f"{r.arch[:5]}, {r.test_site}" for r in R3[R3.aggregation == "logit"].itertuples()], fontsize=6)
axs[2].invert_yaxis(); axs[2].set_xlabel("Cross-source minus pooled", fontsize=7); axs[2].set_title("R3: aggregation rule"); axs[2].legend(frameon=False, fontsize=6); [a_.grid(True, alpha=0.3) for a_ in axs]
fig.tight_layout(w_pad=2.0); save_fig(fig, "fig_s2_robustness")

# 8E: clinical translation (sensitivity at 90% specificity)
fig, ax = plt.subplots(figsize=(COL2, 2.8)); S3 = SENS.reset_index(drop=True); xs = np.arange(len(S3)); cc = {"pooled": "0.35", "pooled x2": "0.75"}
for i, r in S3.iterrows(): ax.bar(i, r.diff_sens_at_90spec, yerr=[[r.diff_sens_at_90spec - r.lo], [r.hi - r.diff_sens_at_90spec]], capsize=2, color=cc[r.comparator], edgecolor="k", lw=0.4)
ax.axhline(0, color="k", lw=0.6); ax.set_xticks(xs); ax.set_xticklabels([f"{r.arch[:5]}, {r.test_site}" for r in S3.itertuples()], rotation=60, ha="right", fontsize=5.5)
ax.set_ylabel("Sensitivity at 90% specificity: cross-source minus comparator"); ax.grid(True, axis="y", alpha=0.3)
from matplotlib.patches import Patch; ax.legend(handles=[Patch(facecolor=cc[k], edgecolor="k", label=f"vs {k}") for k in cc], frameon=False); fig.tight_layout(); save_fig(fig, "fig_s2_clinical_sensitivity")

# 8F: calibration before and after Platt scaling with k labelled target images
fig, axs = plt.subplots(len(TG), len(ARCHS), figsize=(COL2, 6.0), sharex=True)
for r_, T in enumerate(TG):
    for c_, X in enumerate(ARCHS):
        a_ = axs[r_, c_]
        for m_, col in [("pooled", "0.3"), ("cross-source ensemble", "C3")]:
            g = RCS[(RCS.test_site == TG[T]) & (RCS.arch == X) & (RCS.model == m_)].sort_values("K")
            if len(g): a_.plot(g.K, g.ece, "-o", ms=2.5, color=col, label=m_)
        a_.set_xscale("symlog", linthresh=100); a_.set_title(f"{X}, {TG[T]}", fontsize=7); a_.grid(True, alpha=0.3)
        if c_ == 0: a_.set_ylabel("ECE (5 findings)")
for a_ in axs[-1]: a_.set_xlabel("k labelled target images (Platt scaling)")
axs[0, 0].legend(frameon=False, fontsize=6); fig.tight_layout(); save_fig(fig, "fig_s2_recalibration")

In [ ]:
# STAGE 2 / STEP 9 - the decision: verdict, allowed claims, a draft contribution statement filled with YOUR numbers, a literature log and a claim-to-figure map
def rng_txt(x): return "n/a" if len(x) == 0 else (f"{x.min():+.3f} to {x.max():+.3f}" if len(x) > 1 else f"{x.iloc[0]:+.3f}")
pooled_rows = PC[PC.comparator == "pooled"]
numbers = "; ".join(f"{r.arch} on {r.test_site}: {r['diff']:+.3f} (95% CI {r.lo:+.3f} to {r.hi:+.3f})" for _, r in pooled_rows.iterrows())
passed = CELLS[CELLS.passes]; tier = TIER
CLAIM = {"PASS": f"On three public chest X-ray sites, averaging one model trained on each source hospital outperformed a model trained on the pooled data, two pooled models, and two models from the same hospital, "
                 f"in {len(passed)} of {len(CELLS)} architecture-by-site comparisons (difference in mean AUROC versus pooled training {rng_txt(passed['diff_vs_pooled'])}).",
         "PARTIAL": f"The cross-source ensemble clearly beat all comparators in {len(passed)} of {len(CELLS)} architecture-by-site comparisons ({', '.join(passed.arch + ' on ' + passed.test_site) if len(passed) else 'none'}), "
                    "but the advantage did not replicate across architectures and sites; report it as an observation, not as the main finding.",
         "ENSEMBLING_ONLY": f"Averaging two independently trained models beat pooled training and two pooled models in {len(ens_cells)} of {len(CELLS)} architecture-by-site comparisons, but a same-hospital ensemble did as well in most of them: "
                            "the data show a benefit of ensembling, not of hospital diversity.",
         "FAIL": "The cross-source ensemble did not reliably beat pooled training or the controls. Report this as a negative result and make the paper a diagnosis and benchmark study.",
         "REVERSED": "Pooled training beat the cross-source ensemble in at least four comparisons: recommend pooled training and report the ensemble as not worthwhile."}[tier]
ALLOW = {"PASS": ["the headline claim, worded exactly as above with the numbers", "that the gain exceeds plain ensembling (the same-source and pooled x2 controls), if H1 is supported", "a practical recipe: fuse models, then recalibrate with about 500 target labels, if H3 is supported"],
         "PARTIAL": ["the observation for the cells that passed, with their limits", "that results depend on architecture or test site", "no general recommendation"],
         "ENSEMBLING_ONLY": ["that averaging independently trained models beats pooled training here (a known effect, so frame it as a practical finding, not a new method)", "that same-hospital seed ensembles do as well as cross-hospital ensembles", "recalibration as the practical fix for calibration"],
         "FAIL": ["pooled training is a strong baseline that none of the tested alternatives beat", "the diagnosis analyses (asymmetry, case mix, size, calibration)", "recalibration as the practical fix for calibration"],
         "REVERSED": ["pooled training is preferable in this setting", "the diagnosis analyses"]}[tier]
NOT = ["that ensembling under distribution shift is new (it is established in domain generalization)", "that hospital diversity (rather than ensembling in general) explains a gain, unless the same-source controls are beaten", "an unseen-site AUROC drop for VinBigData (it was an easier test site)", "that hospital shift was separated from label-definition shift (MIMIC was not available)",
       "that DANN, CORAL or GroupDRO cannot work (they were not tuned with equal budgets)", "any effect smaller than the minimal detectable difference of these test sets", "findings that rest on Atelectasis, Pneumothorax or Consolidation alone on VinBigData (few positives)"]
NEXT = {"PASS": ["write the paper around the headline claim", "run the gap decomposition and the remaining controls", "replicate on one more site or architecture if possible"],
        "PARTIAL": ["find out why the other cells failed (see H1 and the per-finding results)", "add a fourth site or a third architecture before making the claim", "otherwise write a benchmark paper with this as a secondary result"],
        "ENSEMBLING_ONLY": ["state plainly that hospital diversity gave no extra benefit over same-hospital ensembling", "check H1 (diversity): if cross-hospital pairs are not more diverse, that explains it", "build the paper on diagnosis, the benchmark and the practical recipe (ensemble + recalibration), not on a diversity claim"],
        "FAIL": ["pivot: diagnosis, negative results and recalibration", "tune the DG methods equally and rerun the comparison", "target a benchmark-friendly journal"],
        "REVERSED": ["pivot as for FAIL", "check for a bug or an unfair comparison before accepting the reversal"]}[tier]
mech = f"H1 diversity: {'supported' if H1_PASS else 'not supported'} ({int(H1.cross_more_diverse.sum())}/{len(H1)} cells) | H2 gain vs diversity: rho = {RHO[0]:.2f} [{RHO[1]:.2f}, {RHO[2]:.2f}] ({'supported' if H2_PASS else 'not supported'}) | H3 calibration: ensemble not better calibrated: {'yes' if H3a else 'no'}; Platt at k=500 lowers ECE in every cell: {'yes' if H3b else 'no'}"
md = [f"# Stage 2 decision", f"Generated {datetime.datetime.now():%Y-%m-%d %H:%M}. Plan hash {phash(PLAN)} (locked before the analysis).", "",
      f"## Verdict for the headline claim: **{tier}**", f"{n_pass} of {len(CELLS)} architecture-by-site cells pass; architectures with a pass: {archs_ok}; test sites with a pass: {targets_ok}.", "",
      "## Draft contribution statement (edit only after checking every number)", CLAIM, "", f"Difference versus pooled training, per cell: {numbers}", "",
      "## Mechanism", mech, "", "## Claims you may make"] + [f"- {x}" for x in ALLOW] + ["", "## Claims you must NOT make"] + [f"- {x}" for x in NOT] + ["", "## Next actions"] + [f"- {x}" for x in NEXT]
open(rf"{S2}\stage2_decision.md", "w", encoding="utf-8").write("\n".join(md)); print("\n".join(md))

LIT = pd.DataFrame([
 dict(id=1, reference="Zech et al., PLoS Medicine 2018 (10.1371/journal.pmed.1002683)", checked="web search in this project", shows="CNNs identified the hospital system of a radiograph with ~99.95% accuracy; pooling sites with different prevalence did not improve external performance", implication="measuring the cross-site gap and site separability is not new"),
 dict(id=2, reference="Fernandez-Miranda et al., Scientific Reports 2024", checked="web search in this project", shows="institution, X-ray device and image processing each reduce generalization; device response type mattered most", implication="separate institution from device effects when you discuss causes"),
 dict(id=3, reference="Arpit et al., NeurIPS 2022, Ensemble of Averages", checked="web search in this project", shows="ensembling improves domain generalization over ERM, with a bias-variance explanation", implication="ensembling under shift is not new: your novelty is the cross-hospital setting and the controls"),
 dict(id=4, reference="Federated learning for COVID-19 on CT, external generalisation (Univ. of the Witwatersrand)", checked="web search in this project", shows="global and federated models were near 0.5-0.6 accuracy on a non-participating site", implication="pooled or federated training can fail on an unseen site"),
 dict(id=5, reference="Systematic review of external validation in radiology AI (PMC12689012)", checked="web search in this project", shows="reports that jointly trained multi-institution models beat single-site models on external data", implication="consistent with your pooled-beats-single result; read the primary study"),
 dict(id=6, reference="Pooch et al. 2019, domain shift in chest radiograph classification", checked="NOT re-verified in this project", shows="(from your notes) NIH-trained models generalized worst", implication="verify before citing"),
 dict(id=7, reference="Gulrajani and Lopez-Paz 2021, DomainBed", checked="NOT re-verified in this project", shows="(from memory) tuned ERM is hard to beat in domain generalization", implication="verify before citing")])
Q = pd.DataFrame([dict(database=d, query=q, date_run="", hits="", papers_to_read="", decision="") for d, q in [
 ("PubMed", '("chest radiograph*" OR "chest x-ray*") AND (generalization OR "external validation" OR "domain shift" OR "distribution shift") AND (ensemble OR "model averaging")'),
 ("PubMed", '("chest radiograph*" OR "chest x-ray*") AND (pooled OR "multi-site" OR "multi-institution") AND ("site-specific" OR "single-site") AND ensemble'),
 ("arXiv / Google Scholar", '"chest X-ray" "domain generalization" ensemble hospitals pooled training'),
 ("arXiv / Google Scholar", '"model-level" OR "late fusion" hospital-specific models chest radiograph external validation'),
 ("Scopus / Web of Science", 'TITLE-ABS-KEY("chest radiograph*" AND ("domain generalization" OR "external validation") AND ensemble)')]])
MAP_ = pd.DataFrame([dict(claim=c, figure_or_table=f) for c, f in [
 ("cross-hospital gap and asymmetry", "earlier notebooks: transfer matrix and gap figures"), ("full 3x3 transfer pattern", "fair_ens: fig_transfer_matrix"),
 ("headline: cross-source ensemble vs pooled and controls", "stage2: fig_s2_primary_contrasts, fig_s2_gate_map; fair_ens: fig_paired_contrasts, fig_fair_ensembles_same_architecture"),
 ("adding a source helps", "fair_ens: fig_adding_a_source"), ("mechanism: diversity", "stage2: fig_s2_mechanism"), ("robustness", "stage2: fig_s2_robustness"),
 ("clinical meaning", "stage2: fig_s2_clinical_sensitivity"), ("calibration repair with about 500 labels", "stage2: fig_s2_recalibration"), ("where the models look", "fair_ens: gradcam_* and fig_gradcam_summary")]])
try:
    with pd.ExcelWriter(rf"{S2}\stage2_literature_and_map.xlsx") as w: LIT.to_excel(w, sheet_name="verified_references", index=False); Q.to_excel(w, sheet_name="search_log_to_fill", index=False); MAP_.to_excel(w, sheet_name="claim_to_figure_map", index=False)
except ImportError:
    for nm_, d_ in [("verified_references", LIT), ("search_log_to_fill", Q), ("claim_to_figure_map", MAP_)]: d_.to_csv(rf"{S2}\stage2_{nm_}.csv", index=False)
display(LIT[["id", "reference", "checked", "implication"]]); print("search log template saved: run the 5 queries, fill the table, and read every hit before you claim novelty")

In [ ]:
# STAGE 2 / STEP 10 - save everything
BUN = rf"{OUT}\STAGE2_bundle"; os.makedirs(BUN, exist_ok=True); shutil.copytree(S2, rf"{BUN}\stage2", dirs_exist_ok=True)
json.dump(dict(date=str(datetime.date.today()), plan_hash=phash(PLAN), bootstrap=B, verdict=TIER, reverse=REVERSE, cells_passing=n_pass, inputs=RS, note="analysis plan written to disk before the analysis; results read from the previous notebook's saved logits"), open(rf"{BUN}\settings_stage2.json", "w"), indent=2)
zp = shutil.make_archive(rf"{OUT}\STAGE2_bundle", "zip", BUN); print("bundle:", zp, f"({os.path.getsize(zp) / 1e6:.0f} MB)")
for e in [".csv", ".pdf", ".png", ".md", ".xlsx", ".json"]: print(e, sum(f.endswith(e) for _, _, fs in os.walk(BUN) for f in fs))